# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
# !pip -q install transformers sentencepiece torch   # already installed in this environment

In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























Loading weights: 100%|##########| 76/76 [00:01<00:00, 74.82it/s]


# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


Loading weights: 100%|##########| 190/190 [00:00<00:00, 228.79it/s]
C:\Users\abdul\AppData\Local\Temp\claude\6454c5583654406d\b496dc0a-0b9e-4d45-b22d-9f57bd4a48f5\scratchpad\venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


Loading weights: 100%|##########| 104/104 [00:00<00:00, 173.93it/s]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

### ✅ Solution – Task 1

| # | Task | Best model family | Why |
|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** (BERT, RoBERTa) | The model only has to *understand* the whole sentence and output one label; bidirectional attention sees the full context. |
| 2 | Machine translation | **Encoder–decoder** (T5, BART, MarianMT) | One sequence (source language) must be read completely and turned into a *new* sequence of different length (target language). |
| 3 | Open-ended text generation | **Decoder-only** (GPT family) | It is trained to predict the next token from the previous ones, which is exactly what free-form generation is. |
| 4 | Named Entity Recognition | **Encoder-only** | NER labels every input token; it needs a contextual representation of each token, not new text. |
| 5 | Text summarization | **Encoder–decoder** | The full document is encoded, then a shorter new text is generated from that representation. |

In [6]:
model_families = {
    "Sentiment classification": "Encoder-only",
    "Machine translation": "Encoder-decoder",
    "Open-ended text generation": "Decoder-only",
    "Named Entity Recognition": "Encoder-only",
    "Text summarization": "Encoder-decoder",
}
for task, family in model_families.items():
    print(f"{task:<30} -> {family}")

Sentiment classification       -> Encoder-only
Machine translation            -> Encoder-decoder
Open-ended text generation     -> Decoder-only
Named Entity Recognition       -> Encoder-only
Text summarization             -> Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

### ✅ Solution – Task 2

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

results = classifier(sentences)

for sentence, res in zip(sentences, results):
    print(f"{sentence:<45} -> label: {res['label']:<8} score: {res['score']:.4f}")

The movie was excellent.                      -> label: POSITIVE score: 0.9999
The lecture was difficult to understand.      -> label: NEGATIVE score: 0.9997
I enjoyed learning about transformers.        -> label: POSITIVE score: 0.9997


**Observations**
- *The movie was excellent.* → **POSITIVE (0.9999)**
- *The lecture was difficult to understand.* → **NEGATIVE (0.9997)**
- *I enjoyed learning about transformers.* → **POSITIVE (0.9997)**

All three predictions are correct. The scores are extremely close to 1, which is typical for this model: it was fine-tuned on SST-2 (movie reviews) with only two classes, so even a mild sentence like the lecture one is pushed to a confident NEGATIVE. There is no *neutral* class, and a very high score does not necessarily mean the model is right.

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


### ✅ Solution – Task 3
FLAN-T5 is instruction-tuned, so the task is given as a plain-text instruction (`"Summarize: ..."`). The encoder reads the paragraph, the decoder generates the summary token by token.

In [8]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand, interpret and generate human language. It combines linguistics with machine "
    "learning and deep learning. Modern NLP systems are built on transformer models, which use "
    "attention to capture the relationships between words in a sentence. These systems power "
    "applications such as machine translation, chatbots, search engines, and sentiment analysis "
    "of customer reviews."
)

prompt = "Summarize the following text in one sentence: " + paragraph
inputs = tokenizer(prompt, return_tensors="pt")
summary_ids = model.generate(**inputs, max_new_tokens=50, num_beams=4,
                             no_repeat_ngram_size=3,   # never repeat the same 3 words (prevents loops)
                             early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original length:", len(paragraph.split()), "words")
print("Summary length :", len(summary.split()), "words\n")
print("Summary:", summary)

Original length: 64 words
Summary length : 17 words

Summary: Artificial intelligence is a field of artificial intelligence that combines linguistics with machine learning and deep learning.


**Observations**
- The paragraph (64 words) was reduced to one sentence of 17 words. The summary keeps the key idea that NLP combines linguistics with machine learning and deep learning.
- It is **not perfect**: it says *"Artificial intelligence is a field of artificial intelligence"* instead of *"NLP is a field of AI"*. It also leaves out transformers/attention and the applications. FLAN-T5-**small** has only ~77 M parameters, and errors like this are common for small models; a larger model (flan-t5-base/large, BART-large-CNN) gives more accurate summaries.
- `no_repeat_ngram_size=3` forbids repeating any 3-word sequence. Small models easily get stuck repeating the same phrase, so decoding settings matter as much as the model itself.
- The same weakness is visible in Part B: the English → French example produces a grammatically broken sentence (*"Je s'agissant de la technologie de linguage naturelle"*).


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


### ✅ Solution – Task 5

A chatbot has to *produce* a reply word by word, continuing a conversation, and that is exactly what a decoder-only model is trained to do: predict the next token given all the previous tokens (causal language modelling).
The whole conversation – system instructions, previous turns and the user's new message – can simply be placed in the context as one long prompt, and the model keeps generating the continuation, so no separate encoder is needed.
Because this objective only needs raw text, decoder-only models can be pre-trained on enormous amounts of data and then fine-tuned with instructions and human feedback to behave like an assistant.
This single architecture handles many tasks (answering, explaining, translating, writing code) in a flexible, open-ended way, which is why models like ChatGPT use it.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


### ✅ Solution – Optional Challenge

In [9]:
prompt = "Natural language processing is"

print("=== With set_seed(42): same output every time ===")
for run in range(2):
    set_seed(42)
    out = generator(prompt, max_new_tokens=25, do_sample=True, temperature=0.9, top_k=50)
    print(f"run {run + 1}:", out[0]["generated_text"].replace("\n", " "))

print("\n=== Without set_seed: output changes between runs ===")
for run in range(3):
    out = generator(prompt, max_new_tokens=25, do_sample=True, temperature=0.9, top_k=50)
    print(f"run {run + 1}:", out[0]["generated_text"].replace("\n", " "))

=== With set_seed(42): same output every time ===
run 1: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found
run 2: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found

=== Without set_seed: output changes between runs ===
run 1: Natural language processing is the most widely used language programming language, and several languages are made using it. To learn more about the language, visit http
run 2: Natural language processing is a very different process. It takes a lot of work to take the proper steps to understand grammar or grammar. But if you
run 3: Natural language processing is an automatic process in which the language is built upon by the user. It's used to process the information in an attempt to


**Why do we get different outputs for the same prompt?**

At every step the model outputs a *probability distribution* over the whole vocabulary. With `do_sample=False` (greedy decoding, as in Part A) the most likely token is always chosen, so the output is deterministic.
With `do_sample=True` the next token is **randomly drawn** from that distribution (here restricted to the 50 most likely tokens and shaped by `temperature`), so a different token can be chosen at any step – and once one token differs, everything after it changes too.
`set_seed(42)` fixes the random number generator, which makes the "random" draws repeat exactly; removing it gives a new sample each time.